### Task 1 
Create a ChromaDB collection and convert it into a retriever using LangChain.

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(r"C:\Users\sachi\Downloads\Information_Bulletin.pdf")
documents = loader.load()
print(f"Number of pages: {len(documents)}")


Number of pages: 6


In [4]:
#Split the pdf to chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))


Total Chunks: 33


In [5]:
#create embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings( 
    model_name="all-MiniLM-L6-v2" 
) 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2238.62it/s]


In [ ]:
#Create Chroma vector database
from langchain_chroma import Chroma
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    collection_name="information_bulletin"
)
#Create Dense Retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

### Task 2 
Retrieve the Top 3 document chunks for five different user questions.

In [9]:
questions = [
    "What is the eligibility criteria?",
    "What is the application process?",
    "What documents are required?",
    "What is the exam pattern?",
    "What are the important dates?"
]

for q in questions:
    print("\n" + "="*80)
    print("Question:", q)

    docs = retriever.invoke(q)

    for i, doc in enumerate(docs, 1):
        print(f"\nChunk {i}:")
        print(doc.page_content)
        print("-"*80)


Question: What is the eligibility criteria?

Chunk 1:
● IIMs may verify eligibility at various stages of the selection process, the details of which are 
provided at the website https://iimcat.ac.in. Applicants should note that the mere 
fulfilment of the minimum eligibility criteria will not ensure consideration for shortlisting 
by IIMs. Candidates must declare and maintain a valid and unique email account and a mobile 
phone number throughout the selection process. 
RESERVATION
--------------------------------------------------------------------------------

Chunk 2:
University under Section 3  of  the  UGC  Act,  1956,  or  possess  an  equivalent  qualification   
recognised  by  the  Ministry  of Education, Government of India. 
● Candidates appearing for the final year of Bachelor’s degree/equivalent qualification examination 
and those who have completed degree requirements and are awaiting results can also apply. 
However, it may be noted that such candidates, if selected, wi

In [10]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

In [14]:
questions = [
    "What is the eligibility criteria?",
    "What is the application process?",
    "What documents are required?",
    "What is the exam pattern?",
    "What are the important dates?"
]

for q in questions:
    print("\n" + "="*80)
    print("Question:", q)

    docs = bm25_retriever.invoke(q)

    for i, doc in enumerate(docs, 1):
        print(f"\nChunk {i}:")
        print(doc.page_content)
        print("-"*80)


Question: What is the eligibility criteria?

Chunk 1:
ADMISSION PROCESS  
Please note that each IIM is independent to mandate their own eligibility criteria (including academic 
cut-offs and relative weights) and follow different selection proces ses. Performance in the CAT 2025  
examination is an important component for consideration in the selection process. IIMs may also use 
the previous academic performance of the candidates, relevant work experience and other similar inputs
--------------------------------------------------------------------------------

Chunk 2:
For the purpose of being considered for reservation, the  applicable Central Government list as on the 
last date of CAT registration shall be binding. No subsequent change s will be effective for CAT 
2025 and any subsequent selection process of the IIMs.  
The candidates belonging to the reserved categories need to also note the eligibility requirements 
carefully before applying. It should be noted that while it is 

### Task 3 
Compare the results of: 
- Keyword Search 
- Dense Retrieval
 
Write down which performs better and explain why. 

In [15]:
query = "What is the eligibility criteria?"

print("=" * 80)
print("Query:", query)

# Sparse Retrieval (BM25)
print("\nKeyword Search (BM25)")
bm25_docs = bm25_retriever.invoke(query)

for i, doc in enumerate(bm25_docs, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content)
    print("-" * 80)

# Dense Retrieval (ChromaDB)
print("\nDense Retrieval (ChromaDB)")
dense_docs = retriever.invoke(query)

for i, doc in enumerate(dense_docs, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content)
    print("-" * 80)

Query: What is the eligibility criteria?

Keyword Search (BM25)

Chunk 1:
ADMISSION PROCESS  
Please note that each IIM is independent to mandate their own eligibility criteria (including academic 
cut-offs and relative weights) and follow different selection proces ses. Performance in the CAT 2025  
examination is an important component for consideration in the selection process. IIMs may also use 
the previous academic performance of the candidates, relevant work experience and other similar inputs
--------------------------------------------------------------------------------

Chunk 2:
For the purpose of being considered for reservation, the  applicable Central Government list as on the 
last date of CAT registration shall be binding. No subsequent change s will be effective for CAT 
2025 and any subsequent selection process of the IIMs.  
The candidates belonging to the reserved categories need to also note the eligibility requirements 
carefully before applying. It should be note

### Task 4 
Research BM25 and list two real-world search engines or applications that use keyword-
ranking techniques. 

BM25 (Best Matching 25) is a keyword-ranking algorithm used in information retrieval systems. It ranks documents based on:

- Term Frequency (TF)
- Inverse Document Frequency (IDF)
- Document Length Normalization

It is one of the most widely used ranking algorithms for keyword search.

##### Applications Using BM25

 1. Elasticsearch
- Uses BM25 as the default ranking algorithm.
- Commonly used for enterprise search, e-commerce websites, log analytics, and document search applications.

 2. Apache Solr
- Supports BM25 for ranking search results.
- Widely used for website search, digital libraries, and enterprise information retrieval systems.


### Task 5 
Create a comparison table for: 
- Sparse Retrieval 
- Dense Retrieval 
- BM25 
- Hybrid Search 

Include their advantages, disadvantages, and use cases. 

| Technique | Advantages | Disadvantages | Use Cases |
|------------|------------|---------------|-----------|
| **Sparse Retrieval** | Fast, simple, efficient, low computational cost | Cannot understand semantic meaning, depends on exact keywords | Traditional search engines, document retrieval |
| **Dense Retrieval** | Understands context, handles synonyms, high retrieval accuracy | Requires embeddings, higher computational cost | RAG systems, chatbots, semantic search |
| **BM25** | Excellent keyword ranking, efficient, widely adopted | Misses semantic similarity, relies on keyword overlap | Elasticsearch, Apache Solr, enterprise search |
| **Hybrid Search** | Combines keyword and semantic search, highest accuracy | More complex to implement, higher computational resources | Modern search engines, AI assistants, Retrieval-Augmented Generation (RAG) |